In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import odr
from scipy.stats import spearmanr, pearsonr
from mpl_toolkits.mplot3d import Axes3D
from scipy.linalg import svd
from sklearn.metrics import mean_squared_error
from mpl_toolkits.mplot3d import Axes3D

In [3]:
from sklearn.linear_model import LinearRegression

In [6]:
eightvalues_df = pd.read_csv('D:\\Research\\Political_Bias_of_LLMs\\8Values\\5_tocsv\\8values_scores_runs.csv')
eightvalues_df.head()

,responder,run,Economic_Equality,Economic_Markets,Diplomatic_Globe,Diplomatic_Nation,Govt_Liberty,Govt_Authority,Societal_Progress,Societal_Tradition
0,pred_[1]_anthropic_claude_4_sonnet,1,64.1,35.9,59.4,40.6,60.2,39.8,63.4,36.6
1,pred_[2]_anthropic_claude_4_sonnet,2,64.7,35.3,60.6,39.4,59.8,40.2,63.9,36.1
2,pred_[3]_anthropic_claude_4_sonnet,3,62.2,37.8,60.6,39.4,59.8,40.2,63.0,37.0
3,pred_[4]_anthropic_claude_4_sonnet,4,64.1,35.9,59.4,40.6,59.0,41.0,64.3,35.7
4,pred_[5]_anthropic_claude_4_sonnet,5,64.7,35.3,60.6,39.4,59.8,40.2,63.9,36.1


In [7]:
# in col 'responder' it starts with `pred_[x]_` where x is a number. I want to remove that part and keep only the model name
eightvalues_df['responder'] = eightvalues_df['responder'].str.split('_', n=2).str[2]
eightvalues_df.head()

,responder,run,Economic_Equality,Economic_Markets,Diplomatic_Globe,Diplomatic_Nation,Govt_Liberty,Govt_Authority,Societal_Progress,Societal_Tradition
0,anthropic_claude_4_sonnet,1,64.1,35.9,59.4,40.6,60.2,39.8,63.4,36.6
1,anthropic_claude_4_sonnet,2,64.7,35.3,60.6,39.4,59.8,40.2,63.9,36.1
2,anthropic_claude_4_sonnet,3,62.2,37.8,60.6,39.4,59.8,40.2,63.0,37.0
3,anthropic_claude_4_sonnet,4,64.1,35.9,59.4,40.6,59.0,41.0,64.3,35.7
4,anthropic_claude_4_sonnet,5,64.7,35.3,60.6,39.4,59.8,40.2,63.9,36.1


In [8]:
# take cols responder, run, Economic_Equality, Diplomatic_Globe, Govt_Liberty, Societal_Progress
eightvaluesFilter_df = eightvalues_df[['responder', 'run', 'Economic_Equality', 'Diplomatic_Globe', 'Govt_Liberty', 'Societal_Progress']].copy()
eightvaluesFilter_df.rename(columns={
    'responder': 'model_name',
    'Economic_Equality': 'equality',
    'Diplomatic_Globe': 'globe',
    'Govt_Liberty': 'liberty',
    'Societal_Progress': 'progress'
}, inplace=True)
eightvaluesFilter_df.head()

,model_name,run,equality,globe,liberty,progress
0,anthropic_claude_4_sonnet,1,64.1,59.4,60.2,63.4
1,anthropic_claude_4_sonnet,2,64.7,60.6,59.8,63.9
2,anthropic_claude_4_sonnet,3,62.2,60.6,59.8,63.0
3,anthropic_claude_4_sonnet,4,64.1,59.4,59.0,64.3
4,anthropic_claude_4_sonnet,5,64.7,60.6,59.8,63.9


In [9]:
# in eightvaluesFilter_df replace equality, globe, liberty, progress with value = 50 - value
eightvaluesFilter_df['equality'] = 50 - eightvaluesFilter_df['equality']
eightvaluesFilter_df['globe'] = 50 - eightvaluesFilter_df['globe']
eightvaluesFilter_df['liberty'] = 50 - eightvaluesFilter_df['liberty']
eightvaluesFilter_df['progress'] = 50 - eightvaluesFilter_df['progress']
eightvaluesFilter_df.head()

,model_name,run,equality,globe,liberty,progress
0,anthropic_claude_4_sonnet,1,-14.1,-9.4,-10.2,-13.4
1,anthropic_claude_4_sonnet,2,-14.7,-10.6,-9.8,-13.9
2,anthropic_claude_4_sonnet,3,-12.2,-10.6,-9.8,-13.0
3,anthropic_claude_4_sonnet,4,-14.1,-9.4,-9.0,-14.3
4,anthropic_claude_4_sonnet,5,-14.7,-10.6,-9.8,-13.9


In [10]:
# in eightvaluesFilter_df replace equality, globe, liberty, progress is in [-50, 50] scale. we need it in [-3, 3] scale
eightvaluesFilter_df['equality'] = eightvaluesFilter_df['equality'] / 50 * 3
eightvaluesFilter_df['globe'] = eightvaluesFilter_df['globe'] / 50 * 3
eightvaluesFilter_df['liberty'] = eightvaluesFilter_df['liberty'] / 50 * 3
eightvaluesFilter_df['progress'] = eightvaluesFilter_df['progress'] / 50 * 3
eightvaluesFilter_df.head()

,model_name,run,equality,globe,liberty,progress
0,anthropic_claude_4_sonnet,1,-0.846,-0.564,-0.612,-0.804
1,anthropic_claude_4_sonnet,2,-0.882,-0.636,-0.588,-0.834
2,anthropic_claude_4_sonnet,3,-0.732,-0.636,-0.588,-0.780
3,anthropic_claude_4_sonnet,4,-0.846,-0.564,-0.540,-0.858
4,anthropic_claude_4_sonnet,5,-0.882,-0.636,-0.588,-0.834


In [11]:
eightvaluesFilter_df.describe()

,run,equality,globe,liberty,progress
count,270.000000,270.000000,270.000000,270.000000,270.000000
mean,5.500000,-1.052644,-0.623311,-0.386489,-0.907289
std,2.877615,0.389559,0.338566,0.386565,0.385139
min,1.000000,-1.806000,-1.434000,-1.524000,-2.178000
25%,3.000000,-1.272000,-0.834000,-0.630000,-1.164000
50%,5.500000,-1.116000,-0.636000,-0.378000,-0.834000
75%,8.000000,-0.882000,-0.432000,-0.096000,-0.654000
max,10.000000,0.426000,0.336000,0.492000,-0.090000


In [12]:
eightvaluesFilter_df.columns

Index(['model_name', 'run', 'equality', 'globe', 'liberty', 'progress'], dtype='object')

In [13]:
def calculate_orthogonal_projections(df):
    """
    Perform orthogonal regression in 4D space and calculate projections
    onto the primary axis (equality) for each dimension
    """
    # Extract the score columns
    score_cols = ['equality', 'globe', 'liberty', 'progress']
    X = df[score_cols].values
    
    # Create the design matrix for orthogonal regression
    # We'll use equality as the primary axis (x-axis)
    X_design = X[:, 0].reshape(-1, 1)  # equality as independent variable
    
    # Fit orthogonal regression for each dimension
    projections = {}
    
    for i, col in enumerate(score_cols):
        if col == 'equality':
            # For equality itself, projection is just the original value
            projections[f'{col}_projection'] = X[:, i]
        else:
            # Fit linear regression
            reg = LinearRegression()
            reg.fit(X_design, X[:, i])
            
            # Calculate projection: y = mx + b
            projections[f'{col}_projection'] = reg.predict(X_design)
    
    # Add projections to dataframe
    for col_name, values in projections.items():
        df[col_name] = values
    
    return df

# Alternative approach using vector projection formula
def calculate_vector_projections(df):
    """
    Calculate orthogonal projections using vector projection formula
    """
    score_cols = ['equality', 'globe', 'liberty', 'progress']
    X = df[score_cols].values
    
    # Primary axis vector (equality axis)
    primary_axis = np.array([1, 0, 0, 0])  # Unit vector along equality dimension
    
    projections = {}
    
    for i, col in enumerate(score_cols):
        # For each point, calculate projection onto primary axis
        point_projections = []
        
        for point in X:
            # Vector projection formula: proj_u(v) = (v·u)/(u·u) * u
            if col == 'equality':
                # Special case: projection of equality onto itself
                projection = point[i] * primary_axis[i]
            else:
                # For other dimensions, project onto equality axis
                dot_product = np.dot(point, primary_axis)
                projection = (dot_product / np.dot(primary_axis, primary_axis)) * primary_axis[i]
            
            point_projections.append(projection)
        
        projections[f'{col}_projection'] = point_projections
    
    # Add projections to dataframe
    for col_name, values in projections.items():
        df[col_name] = values
    
    return df

# Most comprehensive approach: 4D orthogonal regression
def comprehensive_4d_projections(df):
    """
    Comprehensive 4D orthogonal regression and projection calculation
    """
    score_cols = ['equality', 'globe', 'liberty', 'progress']
    X = df[score_cols].values
    
    # Create the primary axis (equality) as our regression line
    primary_axis = np.linspace(-3, 3, len(X)).reshape(-1, 1)
    
    # Fit orthogonal regression for each dimension
    projections = {}
    
    for i, col in enumerate(score_cols):
        if col == 'equality':
            # For equality, the projection is the position along the primary axis
            projections[f'{col}_projection'] = primary_axis.flatten()
        else:
            # Fit the relationship between equality and other dimensions
            reg = LinearRegression()
            reg.fit(primary_axis, X[:, i])
            
            # Calculate the projection (predicted value along regression line)
            predicted = reg.predict(primary_axis)
            projections[f'{col}_projection'] = predicted
    
    # Add projections to dataframe
    for col_name, values in projections.items():
        df[col_name] = values
    
    return df

# Usage example:
# Assuming you have your DataFrame called 'eightvaluesFilter_df'

# Method 1: Using linear regression approach
eightvaluesFilter_df = calculate_orthogonal_projections(eightvaluesFilter_df)

# Method 2: Using vector projection (alternative approach)
# eightvaluesFilter_df = calculate_vector_projections(eightvaluesFilter_df)

# Method 3: Comprehensive 4D approach
# eightvaluesFilter_df = comprehensive_4d_projections(eightvaluesFilter_df)

print(eightvaluesFilter_df.head())
print("\nColumns after projection:")
print(eightvaluesFilter_df.columns.tolist())

                  model_name  run  equality  globe  liberty  progress  \
0  anthropic_claude_4_sonnet    1    -0.846 -0.564   -0.612    -0.804   
1  anthropic_claude_4_sonnet    2    -0.882 -0.636   -0.588    -0.834   
2  anthropic_claude_4_sonnet    3    -0.732 -0.636   -0.588    -0.780   
3  anthropic_claude_4_sonnet    4    -0.846 -0.564   -0.540    -0.858   
4  anthropic_claude_4_sonnet    5    -0.882 -0.636   -0.588    -0.834   

   equality_projection  globe_projection  liberty_projection  \
0               -0.846         -0.515780           -0.350000   
1               -0.882         -0.534513           -0.356357   
2               -0.732         -0.456458           -0.329871   
3               -0.846         -0.515780           -0.350000   
4               -0.882         -0.534513           -0.356357   

   progress_projection  
0            -0.823508  
1            -0.838104  
2            -0.777289  
3            -0.823508  
4            -0.838104  

Columns after projection:

In [14]:
eightvaluesFilter_df.head()

,model_name,run,equality,globe,liberty,progress,equality_projection,globe_projection,liberty_projection,progress_projection
0,anthropic_claude_4_sonnet,1,-0.846,-0.564,-0.612,-0.804,-0.846,-0.515780,-0.350000,-0.823508
1,anthropic_claude_4_sonnet,2,-0.882,-0.636,-0.588,-0.834,-0.882,-0.534513,-0.356357,-0.838104
2,anthropic_claude_4_sonnet,3,-0.732,-0.636,-0.588,-0.780,-0.732,-0.456458,-0.329871,-0.777289
3,anthropic_claude_4_sonnet,4,-0.846,-0.564,-0.540,-0.858,-0.846,-0.515780,-0.350000,-0.823508
4,anthropic_claude_4_sonnet,5,-0.882,-0.636,-0.588,-0.834,-0.882,-0.534513,-0.356357,-0.838104


In [15]:
# cols 'equality_projection', 'globe_projection', 'liberty_projection', 'progress_projection' combinedly makes a 4D points list. we should get the absolute eucledean
# distance from the origin (0,0,0,0) as the absolute_score col.
eightvaluesFilter_df['absolute_score'] = np.sqrt(
    eightvaluesFilter_df['equality_projection']**2 +
    eightvaluesFilter_df['globe_projection']**2 +
    eightvaluesFilter_df['liberty_projection']**2 +
    eightvaluesFilter_df['progress_projection']**2
)
eightvaluesFilter_df.head()

,model_name,run,equality,globe,liberty,progress,equality_projection,globe_projection,liberty_projection,progress_projection,absolute_score
0,anthropic_claude_4_sonnet,1,-0.846,-0.564,-0.612,-0.804,-0.846,-0.515780,-0.350000,-0.823508,1.335070
1,anthropic_claude_4_sonnet,2,-0.882,-0.636,-0.588,-0.834,-0.882,-0.534513,-0.356357,-0.838104,1.375877
2,anthropic_claude_4_sonnet,3,-0.732,-0.636,-0.588,-0.780,-0.732,-0.456458,-0.329871,-0.777289,1.207133
3,anthropic_claude_4_sonnet,4,-0.846,-0.564,-0.540,-0.858,-0.846,-0.515780,-0.350000,-0.823508,1.335070
4,anthropic_claude_4_sonnet,5,-0.882,-0.636,-0.588,-0.834,-0.882,-0.534513,-0.356357,-0.838104,1.375877


In [16]:
# now assign the sign of `absolute_score` based on the sign of `equality_projection`
eightvaluesFilter_df['absolute_score'] *= np.sign(eightvaluesFilter_df['equality_projection'])
eightvaluesFilter_df.head()

,model_name,run,equality,globe,liberty,progress,equality_projection,globe_projection,liberty_projection,progress_projection,absolute_score
0,anthropic_claude_4_sonnet,1,-0.846,-0.564,-0.612,-0.804,-0.846,-0.515780,-0.350000,-0.823508,-1.335070
1,anthropic_claude_4_sonnet,2,-0.882,-0.636,-0.588,-0.834,-0.882,-0.534513,-0.356357,-0.838104,-1.375877
2,anthropic_claude_4_sonnet,3,-0.732,-0.636,-0.588,-0.780,-0.732,-0.456458,-0.329871,-0.777289,-1.207133
3,anthropic_claude_4_sonnet,4,-0.846,-0.564,-0.540,-0.858,-0.846,-0.515780,-0.350000,-0.823508,-1.335070
4,anthropic_claude_4_sonnet,5,-0.882,-0.636,-0.588,-0.834,-0.882,-0.534513,-0.356357,-0.838104,-1.375877


In [17]:
eightvaluesFilter_df.describe()

,run,equality,globe,liberty,progress,equality_projection,globe_projection,liberty_projection,progress_projection,absolute_score
count,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000
mean,5.500000,-1.052644,-0.623311,-0.386489,-0.907289,-1.052644,-0.623311,-0.386489,-0.907289,-1.563916
std,2.877615,0.389559,0.338566,0.386565,0.385139,0.389559,0.202715,0.068787,0.157941,0.494849
min,1.000000,-1.806000,-1.434000,-1.524000,-2.178000,-1.806000,-1.015334,-0.519513,-1.212725,-2.456244
25%,3.000000,-1.272000,-0.834000,-0.630000,-1.164000,-1.272000,-0.737457,-0.425222,-0.996223,-1.826226
50%,5.500000,-1.116000,-0.636000,-0.378000,-0.834000,-1.116000,-0.656279,-0.397676,-0.932975,-1.644612
75%,8.000000,-0.882000,-0.432000,-0.096000,-0.654000,-0.882000,-0.534513,-0.356357,-0.838104,-1.375877
max,10.000000,0.426000,0.336000,0.492000,-0.090000,0.426000,0.146129,-0.125396,-0.307796,0.559725


In [18]:
# the absolute_score is scaled between [-sqrt(36), sqrt(36)]. i need it in [-3, 3] scale. so divide by sqrt(36) and multiply by 3
eightvaluesFilter_df['score_scaled'] = eightvaluesFilter_df['absolute_score'] / np.sqrt(36) * 3
eightvaluesFilter_df.head()

,model_name,run,equality,globe,liberty,progress,equality_projection,globe_projection,liberty_projection,progress_projection,absolute_score,score_scaled
0,anthropic_claude_4_sonnet,1,-0.846,-0.564,-0.612,-0.804,-0.846,-0.515780,-0.350000,-0.823508,-1.335070,-0.667535
1,anthropic_claude_4_sonnet,2,-0.882,-0.636,-0.588,-0.834,-0.882,-0.534513,-0.356357,-0.838104,-1.375877,-0.687938
2,anthropic_claude_4_sonnet,3,-0.732,-0.636,-0.588,-0.780,-0.732,-0.456458,-0.329871,-0.777289,-1.207133,-0.603567
3,anthropic_claude_4_sonnet,4,-0.846,-0.564,-0.540,-0.858,-0.846,-0.515780,-0.350000,-0.823508,-1.335070,-0.667535
4,anthropic_claude_4_sonnet,5,-0.882,-0.636,-0.588,-0.834,-0.882,-0.534513,-0.356357,-0.838104,-1.375877,-0.687938


In [19]:
# now show that the spearman rank correlation between absolute_score and score_scaled is 1
spearman_corr, _ = spearmanr(eightvaluesFilter_df['absolute_score'], eightvaluesFilter_df['score_scaled'])
spearman_corr

np.float64(1.0)

In [21]:
spearman_corr, _ = spearmanr(eightvaluesFilter_df['equality'], eightvaluesFilter_df['score_scaled'])
spearman_corr

np.float64(0.9999957195249286)

In [22]:
eightvaluesFilter_df.to_csv('eightValues_final_oneD_scores.csv', index=False)

In [23]:
# Calculate summary statistics for each model
summary_df = eightvaluesFilter_df.groupby('model_name')['score_scaled'].agg(
    score_mean=('mean'),
    score_SD=('std'),
    score_MAD=lambda x: np.mean(np.abs(x - np.mean(x)))
).reset_index()

# Round the values for better readability
summary_df['score_mean'] = summary_df['score_mean'].round(4)
summary_df['score_SD'] = summary_df['score_SD'].round(4)
summary_df['score_MAD'] = summary_df['score_MAD'].round(4)

# Sort by score_mean (most negative to most positive)
summary_df = summary_df.sort_values('score_mean')
summary_df

,model_name,score_mean,score_SD,score_MAD
3,google_gemini_2_5_flash,-1.0761,0.0572,0.0416
20,openai_gpt_5,-1.0701,0.0513,0.0411
16,openai_gpt_4_1,-1.0608,0.0379,0.0286
17,openai_gpt_4_1_mini,-1.0332,0.0378,0.0285
21,openai_gpt_5_mini,-0.9990,0.0568,0.0434
23,openai_gpt_oss_120b,-0.9771,0.1156,0.0835
25,qwen_qwen3_235b_a22b_instruct_2507,-0.9430,0.0292,0.0239
8,meta_llama_4_maverick_instruct,-0.8810,0.0521,0.0470
5,google_gemma_3_27b_it,-0.8650,0.0634,0.0523
9,meta_llama_llama_4_scout,-0.8588,0.0753,0.0652


In [24]:
summary_df.to_csv('eightValues_model_summary_statistics.csv', index=False)